# 00 — Snapshot dos discursos em plenário

Filtra as três arenas, preserva os textos, audita duplicações e realiza a junção temporal.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

DATA_ROOT = Path("/content/drive/MyDrive/falando_nela/data")
REPO_DIR = Path("/content/falando_nela")
REPO_URL = "https://github.com/pedblan/falando_nela.git"
REPO_REF = ""  # Opcional: branch, tag ou commit; vazio acompanha o default remoto.

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--all", "--tags", "--prune"], check=True)
    if not REPO_REF:
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
if REPO_REF:
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_REF], check=True)

os.chdir(REPO_DIR)
os.environ["FALANDO_NELA_DATA_ROOT"] = str(DATA_ROOT)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements-analise.txt"], check=True)
print("Data root:", DATA_ROOT)
print("Commit:", subprocess.run(["git", "rev-parse", "HEAD"], check=True, text=True, capture_output=True).stdout.strip())

## Configuração

Use o mesmo `RUN_ID` em toda a suíte. A configuração versionada é a fonte de verdade.

In [ ]:
from analise.discursos_plenario.config import load_config, resolve_input_paths, resolve_output_root

RUN_ID = "analise-plenario-20260713-v1"
CONFIG_PATH = REPO_DIR / "analise" / "discursos_plenario" / "config.v1.json"
ANALYSIS_CONFIG = load_config(CONFIG_PATH)
RUN_OUTPUT_ROOT = resolve_output_root(ANALYSIS_CONFIG, DATA_ROOT, RUN_ID)
INPUT_PATHS = resolve_input_paths(ANALYSIS_CONFIG, DATA_ROOT)
RODAR_ETAPA = False

assert ANALYSIS_CONFIG.date_start == "2010-02-02"
assert ANALYSIS_CONFIG.date_end == "2026-07-13"
assert ANALYSIS_CONFIG.raw["complete_year_end"] == 2025
assert ANALYSIS_CONFIG.raw["ytd_year"] == 2026
print("Run:", RUN_ID)
print("Saida:", RUN_OUTPUT_ROOT)

## Decisão metodológica

Revise primeiro o inventário de entradas. A etapa é imutável por `RUN_ID`: se dados ou configuração mudarem, crie outro run.

In [ ]:
import pandas as pd
from analise.discursos_plenario.io import input_inventory

SNAPSHOT_INVENTORY = input_inventory(ANALYSIS_CONFIG, DATA_ROOT)
display(SNAPSHOT_INVENTORY)
SNAPSHOT_REQUIRED = ["camara", "senado", "congresso", "parliamentarian_periods"]
SNAPSHOT_MISSING = SNAPSHOT_INVENTORY.loc[
    SNAPSHOT_INVENTORY["entrada"].isin(SNAPSHOT_REQUIRED) & ~SNAPSHOT_INVENTORY["existe"], "caminho"
].tolist()
assert not SNAPSHOT_MISSING, f"Entradas obrigatorias ausentes: {SNAPSHOT_MISSING}"

## Execução

A etapa cara permanece desativada até a inspeção das entradas e dos parâmetros acima.

In [ ]:
from analise.discursos_plenario.snapshot import run_snapshot

SNAPSHOT_RESULT = None
if RODAR_ETAPA:
    SNAPSHOT_RESULT = run_snapshot(
        data_root=DATA_ROOT,
        run_id=RUN_ID,
        config_path=CONFIG_PATH,
        overwrite=False,
    )
    print(SNAPSHOT_RESULT["manifest_path"])
else:
    print("Etapa não executada. Revise o inventário e defina RODAR_ETAPA=True.")

## Validação imediata

Esta checagem não substitui os testes sintéticos nem a revisão dos manifests.

In [ ]:
import pandas as pd

SNAPSHOT_PATH = RUN_OUTPUT_ROOT / "00_snapshot" / "discursos_plenario_snapshot.parquet"
if SNAPSHOT_PATH.exists():
    SNAPSHOT_FRAME = pd.read_parquet(SNAPSHOT_PATH)
    assert set(SNAPSHOT_FRAME["arena"].unique()) <= {"camara", "senado", "congresso"}
    assert SNAPSHOT_FRAME["data_analise"].min() >= pd.Timestamp("2010-02-02")
    assert SNAPSHOT_FRAME["data_analise"].max() <= pd.Timestamp("2026-07-13")
    assert not SNAPSHOT_FRAME.loc[SNAPSHOT_FRAME["ano"].eq(2026), "elegivel_inferencia_anual"].any()
    display(SNAPSHOT_FRAME.groupby(["arena", "ano"]).size().rename("discursos").tail(12))